In [ ]:
import re
import csv
import glob
import json
import torch

from PIL import Image
from tqdm import tqdm
from transformers import GenerationConfig
from gptqmodel import GPTQModel

# load model
# customize load device
load_device = "cuda:0"
torch.cuda.set_device(load_device)
# We take AIDC-AI/Ovis2-34B-GPTQ-Int4 as an example. Note that the code snippet is 
# applicable to any GPTQ-quantized Ovis2 model.
model = GPTQModel.load("AIDC-AI/Ovis2-34B-GPTQ-Int8", device=load_device, trust_remote_code=True)
model.model.generation_config = GenerationConfig.from_pretrained("AIDC-AI/Ovis2-34B-GPTQ-Int8")
text_tokenizer = model.get_text_tokenizer()
visual_tokenizer = model.get_visual_tokenizer()

text = """You are an image analysis tool specialized in facial attribute classification. 
          For the provided face image, output a JSON object with the following attributes:
        
            {
              "gender": ["male", "female"],
              "age": ["young", "middle-aged", "senior"],
              "skin_color": ["light", "medium", "dark"],
              "ancestry": ["asian", "south_asian", "black", "latino/hispanic", "middle_eastern", "white", "indigenous"],
              "hair_color": ["black", "brown", "red", "blonde", "gray", "other"],
              "bangs": ["yes", "no"],
              "bald": ["yes", "no"],
              "beard": ["no", "mustache", "stubble", "full"],
              "glasses": ["no", "regular", "sun"],
              "headwear": ["no", "beanie", "cap", "hat", "headband", "hijab", "helmet", "turban"],
            }

          Ensure the labeling is based on visible evidence only. If an attribute is unclear, return "unknown".
          
          Only output the JSON without any additional explanation or text.
          
          Example JSON output:
          
          {
            "gender": "female",
            "age": "middle-aged",
            "skin_color": "light",
            "ancestry": "asian",
            "hair_color": "black",
            "bangs": "no",
            "bald": "no",
            "beard": "no",
            "glasses": "sun",
            "headwear": "beanie",
          }
          """



In [ ]:
default_keys = ["gender", "age", "skin_color", "ancestry", "hair_color", "bangs", 
                "bald", "beard", "glasses", "headwear"]

# Function to process a single image
def process_images(batch_images):
    batch_input_ids = []
    batch_attention_mask = []
    batch_pixel_values = []
    for image_path in batch_images:
        image = Image.open(image_path)
        query = f'<image>\n{text}'
        prompt, input_ids, pixel_values = model.preprocess_inputs(query, [image], max_partition=9)
        attention_mask = torch.ne(input_ids, text_tokenizer.pad_token_id)
        batch_input_ids.append(input_ids.to(device=model.device))
        batch_attention_mask.append(attention_mask.to(device=model.device))
        batch_pixel_values.append(pixel_values.to(dtype=visual_tokenizer.dtype, device=visual_tokenizer.device))
    batch_input_ids = torch.nn.utils.rnn.pad_sequence([i.flip(dims=[0]) for i in batch_input_ids], batch_first=True,
                                                      padding_value=0.0).flip(dims=[1])
    batch_input_ids = batch_input_ids[:, -model.config.multimodal_max_length:]
    batch_attention_mask = torch.nn.utils.rnn.pad_sequence([i.flip(dims=[0]) for i in batch_attention_mask],
                                                           batch_first=True, padding_value=False).flip(dims=[1])
    batch_attention_mask = batch_attention_mask[:, -model.config.multimodal_max_length:]
    # generate output
    with torch.inference_mode():
        gen_kwargs = dict(
            max_new_tokens=1024,
            do_sample=False,
            top_p=None,
            top_k=None,
            temperature=None,
            repetition_penalty=None,
            eos_token_id=model.generation_config.eos_token_id,
            pad_token_id=text_tokenizer.pad_token_id,
            use_cache=True
        )
        output_ids = model.generate(batch_input_ids, pixel_values=batch_pixel_values, attention_mask=batch_attention_mask,**gen_kwargs)
    result = []
    for i in range(len(batch_images)):
        output = text_tokenizer.decode(output_ids[i], skip_special_tokens=True)
        # Extracting the JSON part
        json_output = re.search(r"\{.*\}", output, re.DOTALL).group()
        json_output = re.sub(r',\s*}', '}', json_output)  # This removes a comma just before a closing curly brace
        data = json.loads(json_output)
        result.append([batch_images[i].replace('../../../data/RFW/data/', '')] + [data.get(key, "unknown") for key in default_keys])
    torch.cuda.empty_cache()
    return result



In [ ]:
init = 0
files = glob.glob('../../../data/RFW/data/*/*/*')

import os
os.makedirs("../../../results/annotations", exist_ok=True)
output_file = "../../../results/annotations/ovis_labels.csv"

# Column names
columns = ["File Name", "Gender", "Age", "Skin Color", "Ancestry", "Hair Color", 
           "Bangs", "Bald", "Beard", "Glasses", "Headwear"]

# Chunk settings
chunk_size = 6  # Number of images processed per batch

# Write CSV header
with open(output_file, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(columns)  # CSV Header

# Process in chunks
for start_idx in tqdm(range(init, len(files), chunk_size), desc="Processing Images"):
    end_idx = min(start_idx + chunk_size, len(files))
    chunk = files[start_idx:end_idx]
    results = process_images(chunk)
    # Save chunk results to CSV
    with open(output_file, "a", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(results)